# 💬 Synthesis Model — Google Colab Notebook
## Retail Analytics Copilot — `LOCAL_MODELS=False`

هذا الـ notebook يشغّل موديل **Synthesis** على Google Colab ويكشفه عبر Ngrok.

**دور الـ Synthesis:** يولّد الإجابة النهائية للمستخدم بدمج:
- نتيجة SQL من قاعدة البيانات
- السياق المسترجع من الوثائق (RAG)
- القيود والتواريخ المستخرجة من الـ Planner

**الموديل المستخدم:** `gemma2:9b-instruct-q5_0` (أكبر وأكثر دقة للإجابات المفصّلة)

---
### ⚠️ تعليمات ما قبل التشغيل:
1. تأكد من تفعيل **GPU Runtime** → Runtime > Change runtime type > T4 GPU
2. ضع `NGROK_AUTH_TOKEN` في Colab Secrets
3. بعد التشغيل، انسخ رابط Ngrok وضعه في `.env` → `NGROK_SYNTHESIS_URL`

In [ ]:
# ─── الخلية 1: إيقاف أي Ollama قديم وتحديث النظام ───────────────────────────
# نوقف عمليات Ollama القديمة ونثبت zstd المطلوبة لفك ضغط نماذج Ollama
!pkill ollama || true
!apt-get update -qq
!apt-get install -y -qq zstd

In [ ]:
# ─── الخلية 2: تثبيت Ollama ────────────────────────────────────────────────
# تحميل وتثبيت Ollama runtime من الموقع الرسمي
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ─── الخلية 3: تشغيل خادم Ollama على المنفذ 8000 ─────────────────────────────
# الـ Synthesis هو أكثر موديل يُستدعى في الـ pipeline لأنه يولّد الإجابة النهائية
# لذا يُشغَّل على منفذ منفصل 8000 بدلاً من 11434 الافتراضي
# OLLAMA_ORIGIN=* يسمح بطلبات Ngrok عبر HTTPS
import os
os.environ["OLLAMA_HOST"] = "127.0.0.1:8000"

!nohup bash -c "OLLAMA_HOST=127.0.0.1:8000 OLLAMA_ORIGIN=* ollama serve" > ollama_synthesis.log 2>&1 &

# انتظار بدء الخادم
!sleep 5

# فحص حالة الخادم
!tail -20 ollama_synthesis.log

In [ ]:
# ─── الخلية 4: تحميل موديل Synthesis ────────────────────────────────────────
# gemma2:9b-instruct-q5_0 = موديل Google Gemma 2 بـ 9 مليار parameter
# q5_0 = تكميم 5-bit للحصول على جودة عالية في توليد النص المفصّل
# حجمه ≈6.4GB — الأبطأ في التحميل لكن الأفضل في جودة الإجابات
# هو الـ Default LM الذي يُضبط في DSPy عند LOCAL_MODELS=False
ollama_model_id = "gemma2:9b-instruct-q5_0"

!OLLAMA_HOST=127.0.0.1:8000 ollama pull {ollama_model_id}

# التأكد من التحميل
!curl -s http://127.0.0.1:8000/api/tags | python3 -c "import json,sys; data=json.load(sys.stdin); [print('✓', m['name']) for m in data.get('models',[])]" 

In [ ]:
# ─── الخلية 5: اختبار توليد إجابة نهائية ────────────────────────────────────
# نختبر قدرة الموديل على دمج بيانات SQL مع السياق وتوليد إجابة مفيدة
%%bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "gemma2:9b-instruct-q5_0",
    "messages": [
      {"role": "user", "content": "Given SQL result: [{\"product\": \"Chai\", \"revenue\": 4887.0}]. Answer: What is the top product by revenue? Provide a brief business insight."}
    ],
    "stream": false
  }' | python3 -c "import json,sys; r=json.load(sys.stdin); print('Answer:', r['choices'][0]['message']['content'])"

In [ ]:
# ─── الخلية 6: تثبيت pyngrok ─────────────────────────────────────────────────
# مكتبة Python للتحكم في Ngrok tunnels برمجياً
!pip install pyngrok -q

In [ ]:
# ─── الخلية 7: تشغيل Ngrok Tunnel ────────────────────────────────────────────
# نقرأ NGROK_AUTH_TOKEN من Colab Secrets لأمان أفضل
# الـ Synthesis tunnel هو الأهم لأن DSPy يستخدمه كـ default LM
from google.colab import userdata
from pyngrok import ngrok, conf

# قراءة token المصادقة من Secrets
ngrok_auth = userdata.get('NGROK_AUTH_TOKEN')
conf.get_default().auth_token = ngrok_auth

# إغلاق tunnels قديمة لتجنب تعارض المنافذ
ngrok.kill()

# فتح tunnel HTTPS على المنفذ 8000
# host_header مطلوب لأن Ollama يرفض طلبات بـ Host غير معروف
tunnel = ngrok.connect(
    8000,
    bind_tls=True,
    host_header="127.0.0.1:8000"
)

print("=" * 60)
print("✅ Synthesis Model جاهز!")
print(f"🌐 NGROK_SYNTHESIS_URL = {tunnel.public_url}")
print("📋 انسخ هذا الرابط وضعه في ملف .env")
print("=" * 60)

In [ ]:
# ─── الخلية 8: اختبار Ngrok بإجابة تركيبية كاملة ─────────────────────────────
# نختبر الـ pipeline الكامل عبر الرابط العام:
# سؤال → بيانات SQL → سياق RAG → إجابة نهائية
import requests

ngrok_url = tunnel.public_url
model_id = ollama_model_id

prompt = """
You are a retail analytics assistant.
SQL Result: [{"category": "Beverages", "revenue": 103924.31}]
Context: According to KPI definitions, revenue = SUM(UnitPrice * Quantity * (1 - Discount)).
Question: What is the revenue for Beverages category?
Give a concise business answer with the exact number.
"""

resp = requests.post(
    f"{ngrok_url}/v1/chat/completions",
    json={
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    },
    timeout=60
)

print(f"Status: {resp.status_code}")
if resp.ok:
    answer = resp.json()["choices"][0]["message"]["content"]
    print(f"✅ Synthesis Answer:\n{answer}")
else:
    print(f"❌ Error: {resp.text}")